## makemore5
#### https://github.com/karpathy/makemore
#### backward()

In [1]:
# 导包
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# 读取输入的文件
words = open('names.txt','r').read().splitlines()
len(words)

32033

In [3]:
# 创建一个映射
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [4]:
# build the dataset
def build_dataset(words):
    block_size = 3
    X,Y = [],[]
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            idx = stoi[ch]
            X.append(context)
            Y.append(idx)
            context = context[1:] + [idx]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape,Y.shape)
    return X,Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [5]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [6]:
n_embd = 10
n_hidden = 200
block_size = 3

g = torch.Generator().manual_seed(42)
C = torch.randn((vocab_size,n_embd),generator=g)

W1 = torch.randn((n_embd*block_size,n_hidden),generator=g) * (5/3)/(n_embd*block_size)**0.5
b1 = torch.randn(n_hidden,generator=g) * 0.1

W2 = torch.randn((n_hidden,vocab_size),generator=g) *0.1
b2 = torch.randn(vocab_size,generator=g) * 0.1

bngain = torch.ones((1,n_hidden))*0.1 + 1.0
bnbias = torch.zeros((1,n_hidden))*0.1

parameters = [C,W1,b1,W2,b2,bngain,bnbias]
sum(p.nelement() for p in parameters)

for p in parameters:
    p.requires_grad = True

In [7]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X, Y

In [8]:
# 向前传播
emb = C[Xb]
embcat = emb.view(emb.shape[0],-1)

hprebn = embcat @ W1  + b1

bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff ** 2
bnvar = 1/(n-1)*(bndiff2).sum(0,keepdim=True)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
    
h = torch.tanh(hpreact)
logits = h @ W2 + b2
# loss = F.cross_entropy(logits,Ytr[ix])
logit_maxes = logits.max(1,keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1,keepdim=True)
counts_sum_inv = counts_sum ** -1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n),Yb].mean()


# 向后传播
for p in parameters:
    p.grad = None
# loss.backward()
for t in [
    logprobs,probs,counts,counts_sum,counts_sum_inv,
    norm_logits,logit_maxes,logits,h,hpreact,bnraw,
    bnvar_inv,bnvar,bndiff2,bndiff,hprebn,bnmeani,embcat,emb
]:
    t.retain_grad()

loss.backward()
loss


tensor(3.9155, grad_fn=<NegBackward0>)

In [9]:
Yb

tensor([ 0, 14,  5, 18, 14,  0,  5,  9,  9,  1, 25,  4,  7,  9, 14, 14, 18, 14,
         9, 18, 14, 14,  0, 20, 12,  5,  5,  3,  1,  4,  0,  6])

In [10]:
logprobs[range(n),Yb]

tensor([-4.7696, -2.5044, -2.5008, -3.3229, -5.2272, -3.4281, -5.4133, -3.8419,
        -4.3091, -4.8532, -3.7134, -4.6408, -4.1541, -3.7338, -4.8742, -2.5690,
        -3.2887, -4.7765, -4.0930, -4.2400, -0.8970, -5.5712, -4.1639, -4.8844,
        -3.2303, -3.8915, -3.4891, -3.6942, -4.6731, -3.6817, -3.7621, -3.1031],
       grad_fn=<IndexBackward0>)

In [15]:
# Exercise 1:backprop through the whole thing manually

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n),Yb] = -1.0/n
dprobs = (1.0 / probs) * dlogprobs
dcounts_sum_div =(counts * dprobs).sum(1,keepdim=True)
dcounts = counts_sum_inv * dprobs
dcounts_sum = (-counts_sum**-2) * dcounts_sum_div
dcounts += torch.ones_like(counts) * dcounts_sum
dnorm_logits = counts * dcounts
dlogits = dnorm_logits.clone()
dlogit_maxes = (-dnorm_logits).sum(1,keepdim = True)
dlogits += F.one_hot(logits.max(1).indices,num_classes = logits.shape[1]) * dlogit_maxes

dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
dhpreact = (1.0 - h**2) *dh

dbngain = (bnraw * dhpreact).sum(0,keepdim  = True)
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0,keepdim = True)
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0,keepdim = True)
dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
dbndiff += (2*bndiff) * dbndiff2

dhprebn = dbndiff.clone()
dbnmeani =(-dbndiff).sum(0)
dhprebn += 1.0/n *(torch.ones_like(hprebn)) * dbnmeani

dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)

demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k,j]
        dC[ix] += demb[k,j] 

cmp('logprobs',dlogprobs,logprobs)
cmp('probs',dprobs,probs)
cmp('counts_sum_inv',dcounts_sum_div,counts_sum_inv)
cmp('counts',dcounts,counts)
cmp('counts_sum',dcounts_sum,counts_sum)
cmp('norm_logits',dnorm_logits,norm_logits)
cmp('logits',dlogits,logits)
cmp('logits_maxes',dlogit_maxes,logit_maxes)

cmp('h',dh,h)
cmp('W2',dW2,W2)
cmp('b2',db2,b2)
cmp('hpreact',dhpreact,hpreact)

cmp('bngain',dbngain,bngain)
cmp('bnbias',dbnbias,bnbias)
cmp('bnraw',dbnraw,bnraw)
cmp('bndiff',dbndiff,bndiff)
cmp('bnvar_inv',dbnvar_inv,bnvar_inv)
cmp('bnvar',dbnvar,bnvar)
cmp('bndiff2',dbndiff2,bndiff2)
cmp('bnmeani',dbnmeani,bnmeani)
cmp('hprebn',dhprebn,hprebn)

cmp('W1',dW1,W1)
cmp('b1',db1,b1)
cmp('embcat',dembcat,embcat)

cmp('emb',demb,emb)
cmp('C',dC,C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
logits_maxes    | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff: